# nb141 — Global activity cliff dataset (pillar 1)

Mine activity cliffs across ALL of ChEMBL — pairs of compounds with Tanimoto >= 0.85 (very similar) but |ΔpEC50| >= 1.0 (>10x activity change).

These cliff transformations encode the medchem 'rules' for where a structural change drastically changes binding. For each cliff pair, record:
  - parent / child SMILES, parent / child pEC50, ΔpEC50
  - target_chembl_id (which receptor)
  - the structural difference (R-group swap via MMP)

Output: a parquet of ~50k–500k global cliff pairs across the ChEMBL universe. Used downstream to train a 'cliff-aware regressor' that predicts when an analog will be a cliff vs flat.

In [ ]:
import os, subprocess, sys
os.environ["PYTHONUNBUFFERED"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit", "tqdm", "pyarrow"], check=False)
import glob, pandas as pd
from pathlib import Path

# Try Kaggle dataset mount (multiple potential paths)
bulk_path = None
for pat in [
    "/kaggle/input/**/chembl_bulk_activities.parquet",
    "/kaggle/input/**/papyrus_full_filtered.parquet",
    "/kaggle/input/**/papyrus_pxr_related_filtered.parquet",
]:
    matches = glob.glob(pat, recursive=True)
    if matches:
        bulk_path = matches[0]
        print(f"Found: {bulk_path}")
        break

if not bulk_path:
    print("No bulk parquet in Kaggle dataset; downloading Papyrus directly...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "papyrus-scripts"], check=False)
    from papyrus_scripts import download_papyrus
    from papyrus_scripts.reader import read_papyrus
    from papyrus_scripts.preprocess import keep_accession
    download_papyrus(version="05.7", only_pp=True, structures=False, descriptors=None)
    TARGETS = ["O75469","Q14994","P11473","Q96RI1","Q13133","P55055","Q07869","P37231","Q03181",
               "P10276","P10826","P13631","P19793","P28702","P48443","P10275","P03372","Q92731",
               "P04150","P08235","P10827","P10828","P41235","P11474","O95718","P62508",
               "P08684","P11712","P33261","P05177","P10635","P05181","P20815","P20813",
               "P08183","Q9UNQ0","Q92887","Q9Y6L6","Q9NPD5","P35869","P02768"]
    chunks = []
    for chunk in read_papyrus(version="05.7", plusplus=True, is3d=False, chunksize=200_000):
        sub = keep_accession(chunk, TARGETS)
        if len(sub) > 0:
            chunks.append(sub)
    df = pd.concat(chunks, ignore_index=True)
    print(f"Downloaded: {len(df):,} rows")
else:
    df = pd.read_parquet(bulk_path)
    print(f"Loaded: {len(df):,} rows  cols={list(df.columns)[:15]}")


In [ ]:
# Normalize columns regardless of source — CAST pchembl to float (may be str from prior save)
def _to_num(s):
    try: return float(s)
    except (ValueError, TypeError): return None

smi_col = 'canonical_smiles' if 'canonical_smiles' in df.columns else ('SMILES' if 'SMILES' in df.columns else df.columns[0])
val_col = 'pchembl_value' if 'pchembl_value' in df.columns else 'pchembl_value_Mean'
tgt_col = 'target_chembl_id' if 'target_chembl_id' in df.columns else 'accession'
print(f'smi={smi_col} val={val_col} tgt={tgt_col}')
df = df.dropna(subset=[smi_col, val_col, tgt_col])
df = df.rename(columns={smi_col:'smi', val_col:'pchembl', tgt_col:'target'})
df['pchembl'] = df['pchembl'].apply(_to_num)
df = df.dropna(subset=['pchembl'])
df = df[['smi','pchembl','target']].copy()
# Median-aggregate (compound, target)
df = df.groupby(['smi','target'])['pchembl'].median().reset_index()
print(f'After per-(compound,target) median agg: {len(df):,} pairs')
print(f'Unique compounds: {df["smi"].nunique():,}, targets: {df["target"].nunique():,}')

In [ ]:
# Find cliff pairs per target: within target, compounds with Tanimoto >= 0.85 but |Δ pchembl| >= 1.0
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import numpy as np, time

TAN_THRESH = 0.85
DELTA_THRESH = 1.0
MIN_COMPOUNDS_PER_TARGET = 20
MAX_PAIRS_PER_TARGET = 50_000  # safety cap per target

# Group by target
tgt_groups = df.groupby('target')
tgt_sizes = tgt_groups.size().sort_values(ascending=False)
print(f'Targets with >= {MIN_COMPOUNDS_PER_TARGET} compounds: {(tgt_sizes >= MIN_COMPOUNDS_PER_TARGET).sum()}')
print(tgt_sizes.head(20))

In [ ]:
def morgan_fp(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048)

cliffs = []
t0 = time.time()
for tgt, grp in tgt_groups:
    if len(grp) < MIN_COMPOUNDS_PER_TARGET:
        continue
    smis = grp['smi'].tolist()
    pvals = grp['pchembl'].values
    fps = [morgan_fp(s) for s in smis]
    valid = [i for i,f in enumerate(fps) if f is not None]
    fps_v = [fps[i] for i in valid]
    smis_v = [smis[i] for i in valid]
    pvals_v = pvals[valid]
    target_cliffs = []
    for i in range(len(fps_v)):
        sims = DataStructs.BulkTanimotoSimilarity(fps_v[i], fps_v[i+1:])
        for j_off, sim in enumerate(sims):
            j = i + 1 + j_off
            if sim >= TAN_THRESH:
                d = abs(pvals_v[i] - pvals_v[j])
                if d >= DELTA_THRESH:
                    target_cliffs.append((tgt, smis_v[i], smis_v[j], pvals_v[i], pvals_v[j], d, sim))
                    if len(target_cliffs) >= MAX_PAIRS_PER_TARGET:
                        break
        if len(target_cliffs) >= MAX_PAIRS_PER_TARGET:
            break
    cliffs.extend(target_cliffs)
    if len(cliffs) % 5000 < 100 and len(cliffs) > 0:
        print(f'  {tgt}: {len(target_cliffs)} cliffs, total {len(cliffs):,}  ({time.time()-t0:.0f}s)')

df_c = pd.DataFrame(cliffs, columns=['target','smi_A','smi_B','pchembl_A','pchembl_B','abs_delta','tanimoto'])
out = Path('/kaggle/working/activity_cliffs_global.parquet')
df_c.to_parquet(out, index=False)
print(f'\nTotal cliffs: {len(df_c):,}')
print(f'By target (top 20):')
print(df_c.groupby('target').size().sort_values(ascending=False).head(20))

In [ ]:
# Summary stats
print(df_c[['abs_delta','tanimoto']].describe())
print(f'\nLarge cliffs (|Δ|>=2): {(df_c["abs_delta"] >= 2).sum():,}')
print(f'Very high sim (>=0.95): {(df_c["tanimoto"] >= 0.95).sum():,}')